# Unit 1 — Hospital Readmission Prediction

Logistic Regression with L2 regularization on the Kaggle Diabetes 130 US Hospitals dataset.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


## 2. Load Kaggle dataset
Download `diabetic_data.csv` from Kaggle and keep it beside this notebook.

In [ ]:
df = pd.read_csv("diabetic_data.csv")
print("Shape:", df.shape)
display(df.head())

## 3. Understand the data

In [ ]:
print(df["readmitted"].value_counts(dropna=False))
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

## 4. Create 30-day readmission target

In [ ]:
df = df[df["readmitted"].notna()].copy()
df["readmission_30"] = (df["readmitted"] == "<30").astype(int)
print(df["readmission_30"].value_counts())
print(df["readmission_30"].value_counts(normalize=True).mul(100).round(2))

## 5. Prepare features

In [ ]:
X = df.drop(columns=["encounter_id","patient_nbr","readmitted","readmission_30"], errors="ignore")
y = df["readmission_30"]
X = X.replace("?", np.nan)
print(X.shape)

## 6. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## 7. Preprocessing pipeline

In [ ]:
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns
cat_cols = X_train.select_dtypes(exclude=["int64","float64"]).columns

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

## 8. Logistic Regression with L2 regularization

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic", LogisticRegression(
        penalty="l2", C=1.0, solver="liblinear",
        max_iter=1000, class_weight="balanced", random_state=42
    ))
])
model.fit(X_train, y_train)
print("Model trained.")

## 9. Evaluate model

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.50).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob),4))
print(classification_report(y_test, y_pred, digits=4))

## 10. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No 30-day readmission","30-day readmission"]).plot()
plt.title("Hospital Readmission Confusion Matrix")
plt.show()
tn, fp, fn, tp = cm.ravel()
print("TN:",tn," FP:",fp," FN:",fn," TP:",tp)

## 11. ROC curve

In [ ]:
fpr,tpr,_ = roc_curve(y_test,y_prob)
plt.figure(figsize=(7,5))
plt.plot(fpr,tpr,label=f"AUC={roc_auc_score(y_test,y_prob):.4f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.grid(); plt.show()

## 12. Threshold analysis

In [ ]:
rows=[]
for t in np.arange(0.20,0.81,0.10):
    p=(y_prob>=t).astype(int)
    rows.append([t,precision_score(y_test,p,zero_division=0),
                 recall_score(y_test,p,zero_division=0),
                 f1_score(y_test,p,zero_division=0)])
threshold_df=pd.DataFrame(rows,columns=["Threshold","Precision","Recall","F1"])
display(threshold_df.round(4))

## 13. Clinical cost discussion

In [ ]:
print("False Negative:", fn)
print("False Positive:", fp)
print("\nFalse negatives can miss patients who may need additional follow-up.")
print("False positives can consume additional clinical resources.")
print("The operating threshold should reflect the relative clinical cost of these errors.")

## 14. Final summary

In [ ]:
print("Logistic Regression + L2")
print("ROC-AUC:",round(roc_auc_score(y_test,y_prob),4))
print("Precision:",round(precision_score(y_test,y_pred,zero_division=0),4))
print("Recall:",round(recall_score(y_test,y_pred,zero_division=0),4))
print("F1:",round(f1_score(y_test,y_pred,zero_division=0),4))

**Kaggle source:** https://www.kaggle.com/datasets/brandao/diabetes